In [ ]:
import os
import time
from dataclasses import dataclass

from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletion

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "").strip()
OPENAI_DEFAULT_BASE_URL = "https://api.openai.com/v1"


@dataclass(frozen=True)
class ModelResponse:
    text: str | None
    refusal: str | None
    requested_model: str
    served_model: str
    finish_reason: str
    prompt_tokens: int | None
    completion_tokens: int | None
    request_id: str | None
    latency_ms: float


class ConnectorError(Exception):
    def __init__(
        self, message: str, *, status_code: int | None = None, request_id: str | None = None
    ) -> None:
        super().__init__(message)
        self.status_code = status_code
        self.request_id = request_id


class ConnectorAuthError(ConnectorError): ...


class ConnectorRequestError(ConnectorError): ...


class ConnectorUnavailableError(ConnectorError): ...


class ConnectorResponseError(ConnectorError): ...


class ConnectorRateLimitError(ConnectorError): ...


class ConnectorOutOfCreditError(ConnectorError): ...


class OpenAIConnector:
    def __init__(
        self,
        *,
        model: str,
        api_key_env: str = "OPENAI_API_KEY",
        base_url: str | None = None,
        timeout_s: float = 30.0,
        max_retries: int = 0,
        http_client=None,
    ) -> None:
        api_key = os.environ.get(api_key_env)
        if not api_key:
            raise ConnectorAuthError(f"environment variable {api_key_env} is not set")

        self.model = model
        self.base_url = base_url or OPENAI_DEFAULT_BASE_URL
        self._client = OpenAI(
            api_key=api_key,
            base_url=self.base_url,
            timeout=timeout_s,
            max_retries=max_retries,
            http_client=http_client,
        )

    def __repr__(self) -> str:
        return f"OpenAIConnector(model={self.model!r}, base_url={self.base_url!r})"

    def generate(self, prompt: str) -> ModelResponse:
        messages = [{"role": "user", "content": prompt}]

        start = time.perf_counter()
        completion = self._client.chat.completions.create(model=self.model, messages=messages)
        latency_ms = (time.perf_counter() - start) * 1000

        choice = completion.choices[0]
        usage = completion.usage

        if not isinstance(completion, ChatCompletion):
            raise ConnectorResponseError(
                f"expected a chat completion, got {type(completion).__name__}"
            )
        if not completion.choices:
            raise ConnectorResponseError(
                "response has no choices", request_id=completion._request_id
            )
        message = completion.choices[0].message
        if message.content is None and message.refusal is None:
            raise ConnectorResponseError(
                "response has neither content nor refusal", request_id=completion._request_id
            )

        return ModelResponse(
            text=choice.message.content,
            refusal=choice.message.refusal,
            requested_model=self.model,
            served_model=completion.model,
            finish_reason=choice.finish_reason,
            prompt_tokens=usage.prompt_tokens if usage else None,
            completion_tokens=usage.completion_tokens if usage else None,
            request_id=completion._request_id,
            latency_ms=latency_ms,
        )

In [ ]:
conn = OpenAIConnector(model=os.environ.get("OPENAI_MODEL"))
conn.generate("What is the capital of France? One word.")